<img src="https://raw.githubusercontent.com/andre-marcos-perez/ebac-course-utils/main/media/logo/newebac_logo_black_half.png" alt="ebac-logo">

---

# **Módulo** | Análise de Dados: Análise Exploratória de Dados de Logística II
Caderno de **Exercícios**<br>
Professor [André Perez](https://www.linkedin.com/in/andremarcosperez/)

---

# **Tópicos**

<ol type="1">
  <li>Manipulação;</li>
  <li>Visualização;</li>
  <li>Storytelling.</li>
</ol>


---

# **Exercícios**

Este *notebook* deve servir como um guia para **você continuar** a construção da sua própria análise exploratória de dados. Fique a vontate para copiar os códigos da aula mas busque explorar os dados ao máximo. Por fim, publique seu *notebook* no [Kaggle](https://www.kaggle.com/).

---

# **Análise Exploratória de Dados de Logística**

## 1\. Contexto

Este projeto tem como objetivo realizar uma Análise Exploratória de Dados (AED) em um conjunto de dados de operações de entrega logística. O foco está em compreender os padrões de tempo de entrega, a distribuição geográfica das operações (origem e destino), e a relação desses fatores com as características das entregas. A análise visa identificar possíveis gargalos, ineficiências ou insights que possam otimizar futuras operações logísticas, como o planejamento de rotas, alocação de recursos e melhoria da eficiência geral do processo de entrega. O conjunto de dados contém informações detalhadas sobre cada solicitação de entrega, incluindo tempos, localizações geográficas e características dos veículos.

## 2\. Pacotes e bibliotecas

In [1]:
# importe todas as suas bibliotecas aqui, siga os padrões do PEP8:
#
# - 1º pacotes nativos do python: json, os, etc.;
# - 2º pacotes de terceiros: pandas, seabornm etc.;
# - 3º pacotes que você desenvolveu.
#
import os
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

...

Ellipsis

## 3\. Exploração de dados

In [11]:
# faça o código de exploração de dados:
#
# - coleta de dados;
# - wrangling da estrutura;
# - exploração do schema;
# - etc.

# faça o código de exploração de dados:
#
# - coleta de dados;
# - wrangling da estrutura;
# - exploração do schema;
# - etc.

# faça o código de exploração de dados:
#
# - coleta de dados;
# - wrangling da estrutura;
# - exploração do schema;
# - etc.

!wget -q "https://raw.githubusercontent.com/andre-marcos-perez/ebac-course-utils/main/dataset/deliveries.json" -O deliveries.json

with open('deliveries.json', 'r') as f:
    data = json.load(f)

if data:
    print(json.dumps(data[0], indent=4)) # Usando json.dumps com indent para melhor legibilidade
else:
    print("A lista de dados está vazia.")

# Extrair coordenadas de origem e entrega
deliveries_df = pd.DataFrame(data)

# Function to safely extract nested data
def get_nested_value(data_dict, keys):
    """Safely retrieves a value from a nested dictionary."""
    if not isinstance(data_dict, dict):
        return None
    value = data_dict
    for key in keys:
        if isinstance(value, dict) and key in value:
            value = value[key]
        else:
            return None
    return value

# Extraia as informações de localização do dicionário 'origin' -> 'coordinates'
# Use the helper function to safely get coordinates
hub_origin_coords = pd.DataFrame([get_nested_value(delivery, ['origin', 'coordinates']) for delivery in data])
# The previous line can result in rows with None if coordinates are missing.
# Let's filter out None or handle them as needed.
# For now, we'll proceed assuming None will be handled later (e.g., NaN in DataFrame)

# Filter out rows where coordinates extraction failed (resulted in None) if necessary,
# or handle potential None values when creating the DataFrame.
# A simpler way is to build a list of dictionaries to pass to DataFrame:
hub_origin_coords_list = []
for delivery in data:
    coords = get_nested_value(delivery, ['origin', 'coordinates'])
    if coords: # Check if coords is not None or empty
        hub_origin_coords_list.append(coords)
    else:
        # Handle cases where coordinates are missing, e.g., append None or a default
        hub_origin_coords_list.append({'latitude': None, 'longitude': None}) # Append None values

hub_origin_coords = pd.DataFrame(hub_origin_coords_list)
hub_origin_coords = hub_origin_coords.rename(columns={'latitude': 'hub_origin_latitude', 'longitude': 'hub_origin_longitude'})


# Extraia as informações de localização do dicionário 'request' -> 'destination' -> 'coordinates'
# Use the helper function to safely get coordinates
deliveries_coords_list = []
for delivery in data:
    coords = get_nested_value(delivery, ['request', 'destination', 'coordinates'])
    if coords: # Check if coords is not None or empty
        deliveries_coords_list.append(coords)
    else:
         # Handle cases where coordinates are missing
        deliveries_coords_list.append({'latitude': None, 'longitude': None}) # Append None values

deliveries_coords = pd.DataFrame(deliveries_coords_list)
deliveries_coords = deliveries_coords.rename(columns={'latitude': 'delivery_latitude', 'longitude': 'delivery_longitude'})

# Remova as colunas aninhadas originais ('origin', 'request', 'deliveries')
# antes de concatenar as novas colunas
# Ensure columns exist before dropping
cols_to_drop = ['origin', 'request', 'deliveries']
deliveries_df = deliveries_df.drop(columns=[col for col in cols_to_drop if col in deliveries_df.columns])


# Junte as novas colunas de coordenadas ao DataFrame principal
deliveries_df = pd.concat([deliveries_df, hub_origin_coords, deliveries_coords], axis=1)

# Exploração do schema
print("Informações do DataFrame:")
deliveries_df.info()

print("\nPrimeiras linhas do DataFrame:")
print(deliveries_df.head())

print("\nEstatísticas descritivas do DataFrame:")
print(deliveries_df.describe())


print("\nVerificando tipos de dados após wrangling:")
# Verificar se as colunas existem antes de tentar acessá-las
if 'request_time' in deliveries_df.columns and 'delivery_time' in deliveries_df.columns:
    print(deliveries_df[['request_time', 'delivery_time']].dtypes)
else:
    print("Colunas 'request_time' ou 'delivery_time' não encontradas.")




{
    "name": "cvrp-2-df-33",
    "region": "df-2",
    "origin": {
        "lng": -48.05498915846707,
        "lat": -15.83814451122274
    },
    "vehicle_capacity": 180,
    "deliveries": [
        {
            "id": "313483a19d2f8d65cd5024c8d215cfbd",
            "point": {
                "lng": -48.11618888384239,
                "lat": -15.848929154862294
            },
            "size": 9
        },
        {
            "id": "320c94b17aa685c939b3f3244c3099de",
            "point": {
                "lng": -48.11819489551,
                "lat": -15.850772371049631
            },
            "size": 2
        },
        {
            "id": "3663b42f4b8decb33059febaba46d5c8",
            "point": {
                "lng": -48.11248339849675,
                "lat": -15.84787055941764
            },
            "size": 1
        },
        {
            "id": "e11ab58363c38d6abc90d5fba87b7d7",
            "point": {
                "lng": -48.11802268617869,
                "la

## 4\. Manipulação

In [13]:
# faça o código de manipulação de dados:
#
# - enriquecimento;
# - controle de qualidade;
# - etc.

# faça o código de manipulação de dados:
#
# - enriquecimento;
# - controle de qualidade;
# - etc.


# 1. Controle de Qualidade: Verificar e tratar valores ausentes
print("Verificando valores nulos por coluna antes do tratamento:")
print(deliveries_df.isnull().sum())

# Neste dataset específico, parece que não há valores nulos nas colunas de coordenadas
# após a correção anterior que adiciona None. Se houver, você pode decidir como tratá-los:
# - Remover linhas com nulos: deliveries_df.dropna(subset=['coluna1', 'coluna2'], inplace=True)
# - Preencher nulos: deliveries_df.fillna(valor_ou_metodo, inplace=True)
# Para este exemplo, vamos assumir que None nas coordenadas serão tratados como NaN pelo pandas
# e não precisam de tratamento adicional por enquanto, a menos que a análise subsequente exija.

# 2. Transformação: Converter colunas de tempo para datetime
# As colunas de tempo estão em timestamp (segundos desde a época)
# Vamos converter para o tipo datetime do pandas para facilitar cálculos de tempo

# Verifica se as colunas existem antes de tentar convertê-las
if 'request_time' in deliveries_df.columns:
    deliveries_df['request_time'] = pd.to_datetime(deliveries_df['request_time'], unit='s')
    print("\nColuna 'request_time' convertida para datetime.")
else:
    print("\nColuna 'request_time' não encontrada para conversão.")

if 'delivery_time' in deliveries_df.columns:
    deliveries_df['delivery_time'] = pd.to_datetime(deliveries_df['delivery_time'], unit='s')
    print("Coluna 'delivery_time' convertida para datetime.")
else:
     print("Coluna 'delivery_time' não encontrada para conversão.")

# Verificando os tipos de dados após a conversão
print("\nTipos de dados após conversão de tempo:")
# Adiciona um check para garantir que as colunas existem antes de tentar acessar .dtypes
cols_to_check_dtypes = [col for col in ['request_time', 'delivery_time'] if col in deliveries_df.columns]
if cols_to_check_dtypes:
    print(deliveries_df[cols_to_check_dtypes].dtypes)
else:
    print("Colunas 'request_time' ou 'delivery_time' não encontradas para verificar tipos.")

# 3. Enriquecimento: Calcular o tempo total de entrega
# Vamos calcular a diferença entre delivery_time e request_time
# Certifique-se de que ambas as colunas foram convertidas com sucesso

if 'request_time' in deliveries_df.columns and 'delivery_time' in deliveries_df.columns:
    # Calcula a diferença (resulta em um objeto Timedelta)
    deliveries_df['delivery_elapsed_time'] = deliveries_df['delivery_time'] - deliveries_df['request_time']
    print("\nColuna 'delivery_elapsed_time' (tempo total de entrega) adicionada.")

    # Se quiser a duração em segundos, minutos ou horas:
    deliveries_df['delivery_elapsed_seconds'] = deliveries_df['delivery_elapsed_time'].dt.total_seconds()
    print("Coluna 'delivery_elapsed_seconds' adicionada.")

    # Exemplo: converter para minutos
    deliveries_df['delivery_elapsed_minutes'] = deliveries_df['delivery_elapsed_seconds'] / 60
    print("Coluna 'delivery_elapsed_minutes' adicionada.")

    # Verificando as novas colunas
    # Adiciona um check para garantir que as colunas existem antes de tentar acessá-las
    cols_to_print_head = [col for col in ['request_time', 'delivery_time', 'delivery_elapsed_time', 'delivery_elapsed_seconds', 'delivery_elapsed_minutes'] if col in deliveries_df.columns]
    if cols_to_print_head:
        print("\nPrimeiras linhas com as novas colunas de tempo:")
        print(deliveries_df[cols_to_print_head].head())
    else:
         print("\nNenhuma das colunas de tempo esperadas encontrada para imprimir o cabeçalho.")


    # Adiciona um check para garantir que a coluna existe antes de tentar descrever
    if 'delivery_elapsed_minutes' in deliveries_df.columns:
        print("\nEstatísticas descritivas para o tempo de entrega em minutos:")
        print(deliveries_df['delivery_elapsed_minutes'].describe())
    else:
        print("\nColuna 'delivery_elapsed_minutes' não encontrada para estatísticas descritivas.")


else:
    print("\nNão foi possível calcular o tempo de entrega. Verifique as colunas 'request_time' e 'delivery_time'.")


# 4. Controle de Qualidade: Verificar se existem tempos de entrega negativos
# Tempos de entrega negativos podem indicar erros nos dados
if 'delivery_elapsed_seconds' in deliveries_df.columns:
    negative_delivery_times = deliveries_df[deliveries_df['delivery_elapsed_seconds'] < 0]
    if not negative_delivery_times.empty:
        print(f"\nATENÇÃO: Foram encontrados {len(negative_delivery_times)} registros com tempo de entrega negativo!")
        print("Exemplos de registros com tempo de entrega negativo:")
        print(negative_delivery_times[['request_time', 'delivery_time', 'delivery_elapsed_seconds']].head())
        # Decida como tratar esses casos: remover, investigar, etc.
        # Exemplo: remover (se forem poucos e não representarem um padrão)
        # deliveries_df = deliveries_df[deliveries_df['delivery_elapsed_seconds'] >= 0].copy()
        # print("Registros com tempo de entrega negativo removidos.")
    else:
        print("\nNenhum registro com tempo de entrega negativo encontrado.")
else:
     print("\nNão foi possível verificar tempos de entrega negativos. Coluna 'delivery_elapsed_seconds' não encontrada.")


# Exemplo de controle de qualidade adicional: verificar duplicatas
print(f"\nNúmero de linhas duplicadas: {deliveries_df.duplicated().sum()}")
# Se houver duplicatas, você pode removê-las:
# deliveries_df.drop_duplicates(inplace=True)
# print("Linhas duplicadas removidas.")


# Exemplo de controle de qualidade adicional: verificar valores únicos em colunas categóricas
# print("\nValores únicos na coluna 'region':")
# print(deliveries_df['region'].unique())
# print("\nContagem de valores na coluna 'region':")
# print(deliveries_df['region'].value_counts())
# Faça isso para outras colunas categóricas relevantes.


# Exemplo de controle de qualidade adicional: verificar ranges de valores em colunas numéricas
# print("\nVerificando faixas de valores para latitudes e longitudes:")
# print(deliveries_df[['hub_origin_latitude', 'hub_origin_longitude', 'delivery_latitude', 'delivery_longitude']].agg(['min', 'max']))
# Isso pode ajudar a identificar valores muito fora do esperado (erros de digitação, etc.)

print("\nManipulação de dados concluída. DataFrame atualizado:")
print(deliveries_df.info())

Verificando valores nulos por coluna antes do tratamento:
name                      0
region                    0
vehicle_capacity          0
hub_origin_latitude     199
hub_origin_longitude    199
delivery_latitude       199
delivery_longitude      199
dtype: int64

Coluna 'request_time' não encontrada para conversão.
Coluna 'delivery_time' não encontrada para conversão.

Tipos de dados após conversão de tempo:
Colunas 'request_time' ou 'delivery_time' não encontradas para verificar tipos.

Não foi possível calcular o tempo de entrega. Verifique as colunas 'request_time' e 'delivery_time'.

Não foi possível verificar tempos de entrega negativos. Coluna 'delivery_elapsed_seconds' não encontrada.

Número de linhas duplicadas: 0

Manipulação de dados concluída. DataFrame atualizado:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199 entries, 0 to 198
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0  

## 5\. Visualização

In [16]:
# faça o código de visualização de dados:
#
# - produza pelo menos duas visualizações;
# - adicione um pequeno texto com os insights encontrados;
# - etc.

# faça o código de visualização de dados:
#
# - produza pelo menos duas visualizações;
# - adicione um pequeno texto com os insights encontrados;
# - etc.

# --- Passo de Debugging: Inspecionar o DataFrame antes de plotar ---
print("Inspecionando deliveries_df antes da visualização:")
print(deliveries_df.info())
print("\nPrimeiras 5 linhas:")
print(deliveries_df.head())
print("-" * 30)
# --- Fim do Passo de Debugging ---


# 1. Distribuição do Tempo Total de Entrega em Minutos
print("Visualizando a distribuição do tempo total de entrega:")

# Adiciona um check para garantir que a coluna existe antes de tentar plotar
if 'delivery_elapsed_minutes' in deliveries_df.columns:
    print("Coluna 'delivery_elapsed_minutes' encontrada. Prosseguindo com o gráfico.") # Mensagem de debug adicionada
    plt.figure(figsize=(10, 6))
    sns.histplot(deliveries_df['delivery_elapsed_minutes'], bins=50, kde=True)
    plt.title('Distribuição do Tempo Total de Entrega em Minutos')
    plt.xlabel('Tempo de Entrega (minutos)')
    plt.ylabel('Frequência')
    plt.grid(axis='y', alpha=0.75)
    plt.show() # Garante que o gráfico é exibido

    # Insights:
    print("\nInsights da Distribuição do Tempo de Entrega:")
    # Você precisa substituir o texto placeholder pelas suas observações reais do gráfico
    print("A maioria das entregas parece ser concluída em torno de X a Y minutos (observar pico no histograma).")
    print("Há uma cauda mais longa para tempos de entrega maiores, indicando algumas entregas que levam consideravelmente mais tempo.")
    print("A forma da distribuição (simétrica, assimétrica) e a presença de múltiplos picos (se houver) podem sugerir diferentes padrões operacionais ou regiões.")

else:
    # Mensagem modificada para debug
    print("Coluna 'delivery_elapsed_minutes' não encontrada para visualização da distribuição. Verifique a saída de 'deliveries_df.info()' acima.")


# 2. Distribuição Geográfica dos Hubs de Origem e Destinos de Entrega
print("\nVisualizando a distribuição geográfica dos hubs e destinos:")

# Filtra linhas onde as coordenadas não são nulas para evitar erros no plot
geo_df = deliveries_df.dropna(subset=['hub_origin_latitude', 'hub_origin_longitude', 'delivery_latitude', 'delivery_longitude']).copy()

# --- Passo de Debugging: Verificar o conteúdo de geo_df ---
print(f"Número de linhas em geo_df após dropna: {len(geo_df)}") # Mensagem de debug adicionada
# --- Fim do Passo de Debugging ---


# Verifica se o DataFrame de dados geográficos não está vazio após filtrar nulos
if not geo_df.empty:
    print("DataFrame geo_df não está vazio. Prosseguindo com o gráfico geográfico.") # Mensagem de debug adicionada
    plt.figure(figsize=(12, 8))

    # Plotar hubs de origem
    sns.scatterplot(
        x='hub_origin_longitude',
        y='hub_origin_latitude',
        data=geo_df,
        color='red',
        label='Hubs de Origem',
        alpha=0.6,
        s=50 # Tamanho dos pontos
    )

    # Plotar destinos de entrega
    sns.scatterplot(
        x='delivery_longitude',
        y='delivery_latitude',
        data=geo_df,
        color='blue',
        label='Destinos de Entrega',
        alpha=0.3,
        s=10 # Tamanho dos pontos
    )

    plt.title('Distribuição Geográfica dos Hubs de Origem e Destinos de Entrega')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.legend()
    plt.grid(True)
    plt.show() # Garante que o gráfico é exibido

    # Insights:
    print("\nInsights da Distribuição Geográfica:")
    # Você precisa substituir o texto placeholder pelas suas observações reais do gráfico
    print("Os hubs de origem (pontos vermelhos maiores) estão concentrados em determinadas áreas.")
    print("Os destinos de entrega (pontos azuis menores) mostram a área de cobertura das entregas.")
    print("Observe se os destinos se espalham uniformemente a partir dos hubs ou se há clusters em outras regiões.")
    print("A sobreposição e o alcance dos pontos azuis a partir dos vermelhos indicam as rotas e áreas de serviço.")
else:
    # Mensagem modificada para debug
    print("\nNão há dados geográficos válidos para visualização. Verifique a saída de 'deliveries_df.info()' e o tamanho de 'geo_df' acima.")

# Lembre-se de substituir os textos de insights com suas próprias observações baseadas nos gráficos gerados.




Inspecionando deliveries_df antes da visualização:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199 entries, 0 to 198
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   name                  199 non-null    object
 1   region                199 non-null    object
 2   vehicle_capacity      199 non-null    int64 
 3   hub_origin_latitude   0 non-null      object
 4   hub_origin_longitude  0 non-null      object
 5   delivery_latitude     0 non-null      object
 6   delivery_longitude    0 non-null      object
dtypes: int64(1), object(6)
memory usage: 11.0+ KB
None

Primeiras 5 linhas:
           name region  vehicle_capacity hub_origin_latitude  \
0  cvrp-2-df-33   df-2               180                None   
1  cvrp-2-df-73   df-2               180                None   
2  cvrp-2-df-20   df-2               180                None   
3  cvrp-1-df-71   df-1               180                None  